# Information Retrieval — Corpus Builder
Collects documents from Wikipedia, arXiv, AI generation (Ollama), and Dev.to blogs, then merges them into a single `corpus.csv`.

## Topic Map
Defines the 8 topics and their associated search queries used across all data sources.

In [ ]:
topics = {
    'AI': 'artificial intelligence',
    'Cybersecurity': 'cybersecurity network security',
    'Blockchain': 'blockchain cryptocurrency',
    'Cloud Computing': 'cloud computing infrastructure',
    'NLP': 'natural language processing',
    'Robotics': 'robotics autonomous systems',
    'Quantum Computing': 'quantum computing',
    'Data Science': 'data science machine learning'
}

## Part 1 — Wikipedia Articles (Long-form, 2000+ words)
Uses the `wikipedia` and `wikipediaapi` libraries to search and retrieve full article content.
Only articles with more than 2000 words are kept. Collects up to 16 articles per topic, capped at 125 total.

In [ ]:
# Wikipedia Articles/Long form content
import wikipedia 
import wikipediaapi 
import pandas as pd

wiki = wikipediaapi.Wikipedia(
    user_agent='MyProject/1.69 (NoWayJose@exlololol.com)',
    language='en',
)


records = []
for topic, query in topics.items():
    collected = 0
    search_results = wikipedia.search(query, results=100)

    for title in search_results:
        if collected >= 16:
            break

        if len(records) >= 125:
            break

        try:
            page = wikipedia.page(title, auto_suggest=False)
            text = page.content
            word_count = len(text.split())

            if word_count < 2001:
                continue

            records.append({
                'title': page.title,
                'text': text,
                'url': page.url,
                'content_type': 'wikipedia',
                'topic': topic,
                'word_count': word_count
            })
            collected += 1
            print(f"[{topic}] {collected}/16 — {page.title} ({word_count} words)")

        except Exception as e:
            print(f"Skipping '{title}': {e}")
            continue

    if len(records) >= 125:
        break

df_wiki = pd.DataFrame(records)
df_wiki.to_csv('wikipedia_long.csv', index=False)
print(f"\nTotal collected: {len(df_wiki)} documents")

## Part 2 — arXiv Abstracts (Short-form, under 300 words)
Uses the `arxiv` library to fetch paper abstracts.
Only abstracts with fewer than 300 words are kept. Collects up to 16 per topic, capped at 125 total.

In [ ]:
# Arxiv Documents (300 words or less)
import arxiv
records = []
client = arxiv.Client()

for topic, query in topics.items():
    collected = 0
    search = arxiv.Search(query=query, max_results=100)

    for result in client.results(search):
        if collected >= 16:
            break

        if len(records) >= 125:  
            break

        text = result.summary
        word_count = len(text.split())

        if word_count >= 300:
            continue

        records.append({
            'title': result.title,
            'text': text,
            'url': result.entry_id,
            'content_type': 'arxiv_abstract',
            'topic': topic,
            'word_count': word_count
        })
        collected += 1
        print(f"[{topic}] {collected}/16 — {result.title} ({word_count} words)")

    if len(records) >= 125:
        break

df_arxiv = pd.DataFrame(records)
df_arxiv.to_csv('arxiv_short.csv', index=False)
print(f"\nTotal collected: {len(df_arxiv)} documents")

## Part 3 — AI-Generated Documents (Medium-form, 300–2000 words)
Uses a local Llama 3.2 model via `langchain-ollama` to generate technical documents.
Each of the 24 prompts maps to one of the 8 topics. Requires Ollama running locally with the `llama3.2:1b` model pulled.

In [ ]:
# Ai Generated Documents (Medium length, 301-1999)
import pandas as pd
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

model = OllamaLLM(model="llama3.2:1b")

template = """
Write a detailed technical document about the following topic: {topic}
The document should be informative, well-structured, and between 300 and 2000 words.
Only return the document text, no extra commentary.
"""
prompt = ChatPromptTemplate.from_template(template)
chain = prompt | model

topics_ai = [
    "How neural networks learn using backpropagation",
    "The role of transformers in modern NLP",
    "Reinforcement learning and its real-world applications",
    "How phishing attacks are carried out and prevented",
    "The fundamentals of zero-trust security architecture",
    "Common vulnerabilities in web applications",
    "How consensus mechanisms work in blockchain",
    "The use of smart contracts in decentralized finance",
    "Blockchain scalability challenges and solutions",
    "How containerization works with Docker and Kubernetes",
    "Serverless computing explained",
    "Cloud storage architecture and reliability",
    "How sentiment analysis works in NLP",
    "Named entity recognition techniques",
    "Machine translation and its challenges",
    "How SLAM works in autonomous robots",
    "The role of computer vision in robotics",
    "Human-robot interaction and safety",
    "Quantum entanglement and its role in computing",
    "How quantum error correction works",
    "Quantum cryptography explained",
    "Feature engineering in machine learning pipelines",
    "How gradient boosting algorithms work",
    "Data preprocessing and cleaning techniques"
]

topic_map = {
    "How neural networks learn using backpropagation": "AI",
    "The role of transformers in modern NLP": "AI",
    "Reinforcement learning and its real-world applications": "AI",
    "How phishing attacks are carried out and prevented": "Cybersecurity",
    "The fundamentals of zero-trust security architecture": "Cybersecurity",
    "Common vulnerabilities in web applications": "Cybersecurity",
    "How consensus mechanisms work in blockchain": "Blockchain",
    "The use of smart contracts in decentralized finance": "Blockchain",
    "Blockchain scalability challenges and solutions": "Blockchain",
    "How containerization works with Docker and Kubernetes": "Cloud Computing",
    "Serverless computing explained": "Cloud Computing",
    "Cloud storage architecture and reliability": "Cloud Computing",
    "How sentiment analysis works in NLP": "NLP",
    "Named entity recognition techniques": "NLP",
    "Machine translation and its challenges": "NLP",
    "How SLAM works in autonomous robots": "Robotics",
    "The role of computer vision in robotics": "Robotics",
    "Human-robot interaction and safety": "Robotics",
    "Quantum entanglement and its role in computing": "Quantum Computing",
    "How quantum error correction works": "Quantum Computing",
    "Quantum cryptography explained": "Quantum Computing",
    "Feature engineering in machine learning pipelines": "Data Science",
    "How gradient boosting algorithms work": "Data Science",
    "Data preprocessing and cleaning techniques": "Data Science"
}


records = []
for topic in topics_ai:
    print(f"Generating: {topic}")
    result = chain.invoke({"topic": topic})
    word_count = len(result.split())
    records.append({
        'title': topic,
        'text': result,
        'url': 'AI-generated',
        'content_type': 'AI-generated',
        'topic': topic_map[topic],
        'word_count': word_count
    })
    print(f"  Done — {word_count} words")


df_ai = pd.DataFrame(records)
df_ai.to_csv('ai_generated.csv', index=False)
print(f"Total AI documents: {len(df_ai)}")

## Part 4 — Technical Blog Posts from Dev.to (Medium-form, 300–2000 words)
Hits the public Dev.to API to fetch articles by topic tag.
For each article, a second request fetches the full markdown body.
Only posts between 300 and 2000 words are kept. Collects up to 16 per topic.

In [ ]:
# Technical Blog Posts (Medium length, 301-1999 words)
import requests
import pandas as pd
import time

records = []
for topic, query in topics.items():
    print(f"Fetching Dev.to posts for: {topic}")
    
    # API endpoint for searching by tag/query
    url = f"https://dev.to/api/articles?tag={topic.lower().replace(' ', '')}&per_page=30"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error fetching {topic}: {response.status_code}")
        continue
        
    articles = response.json()
    collected = 0
    
    for article in articles:
        if collected >= 16: break
        
        # We need to fetch the full content (the 'body_markdown')
        detail_url = f"https://dev.to/api/articles/{article['id']}"
        detail_resp = requests.get(detail_url)
        
        if detail_resp.status_code == 200:
            content = detail_resp.json().get('body_markdown', '')
            word_count = len(content.split())
            
            # Filter for "Medium" length (300 - 2000 words)
            if 300 <= word_count <= 2000:
                records.append({
                    'title': article['title'],
                    'text': content,
                    'url': article['url'],
                    'content_type': 'technical_blog',
                    'topic': topic,
                    'word_count': word_count
                })
                collected += 1
                print(f"  [{collected}/16] {article['title']} ({word_count} words)")
        
        time.sleep(0.5) # Be kind to the API

df_blogs = pd.DataFrame(records)
df_blogs.to_csv('devto_medium.csv', index=False)
print(f"Total Blog Documents: {len(df_blogs)}")

## Part 5 — Merge All Sources into corpus.csv
Reads the four saved CSVs and concatenates them into a single unified corpus.
Prints the total document count and a breakdown by content type.

In [ ]:
# Compiling the Documents Together
df_wiki_long = pd.read_csv('wikipedia_long.csv')
df_arxiv = pd.read_csv('arxiv_short.csv')
df_ai = pd.read_csv('ai_generated.csv')
df_blogs = pd.read_csv('devto_medium.csv')

df_all = pd.concat([df_wiki_long, df_arxiv, df_ai, df_blogs], ignore_index=True)
df_all.to_csv('corpus.csv', index=False)
print(f"Total combined documents: {len(df_all)}")
print(df_all['content_type'].value_counts())